# MC-dropout — bất định epistemic từ 5 checkpoint đã có

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

**Không train gì cả.** Notebook này chỉ chạy inference: nạp `best.pt` của từng fold,
bật lại dropout, forward `K` lượt trên tập val của chính fold đó, lưu `(K, N, 7)`.

**Vì sao không dùng thẳng 5 checkpoint làm deep ensemble.** Mỗi ca ở val của fold `f`
nằm trong tập train của **cả 4 model kia** (kiểm trên `splits/`, WORKLOG S-080). Gộp
5 model rồi chấm trên 394 ca là để 4/5 thành viên chấm bài họ đã học thuộc. MC-dropout
né đúng chỗ đó: mọi thành viên đều là cùng một model của fold đó, nên đều mù với val.

**Đổi lại:** MC-dropout là xấp xỉ nghèo hơn deep ensemble thật — các thành viên chung
một cực tiểu nên đa dạng ít. Đây là phép đo rẻ để quyết có đáng đốt 4 session Kaggle
cho ensemble nhiều seed hay không, chứ không phải bản thay thế.

**Ngân sách:** ~8 phút GPU cho cả 5 fold ở `K=20`. So với 37.5h của ensemble 3 seed.

**Cần mount hai dataset:** cache E4, và checkpoint (`best-weights`, 5 file `best_fold_N.pt`).

## 0. Bootstrap

Dòng `repo commit` là bằng chứng đang chạy đúng bản code nào.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1, 2, 3, 4, 5]
N_PASSES = 20          # số lượt forward mỗi ca
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

os.environ["LLDMMRI_OUTPUT_DIR"] = "/kaggle/working/runs/mc_dropout"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / "baseline_3dpatch.yaml"
CFG = load_yaml(CFG_PATH)
print("dropout_prob trong config:", CFG["model"].get("dropout_prob"))

## 1. Cache E4 và checkpoint

Cần **hai** thứ mount vào: cache E4 (`lesion_tight · 112×112×32 · per_phase`) và 5 file
`best.pt`. Đổi đường dẫn bên dưới cho khớp tên dataset bạn đã upload.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

# Cache E4 nhận diện bằng NỘI DUNG `cache_meta.json`, không bằng tên dataset. Tên do
# người upload đặt và đã lệch một lần rồi (`lld-mmri-lesion-tight/cache_lesion_tight`
# chứ không phải `lld-mmri-e4-per-phase` như đoán ở S-080). Ba khoá này là thứ phân
# biệt E4 với mọi cache trước đó.
E4_KEYS = {
    "align_phases": "per_phase",          # <- phân biệt E4 với E3
    "target_size": [112, 112, 32],        # <- phân biệt E3/E4 với E0/E1
    "crop_mode": "lesion_tight",          # <- phân biệt E1+ với E0
}

# Tên file checkpoint. KHÔNG kèm thư mục cha — độ sâu do `rglob` lo, xem bên dưới.
#   A) best_fold_1.pt ... best_fold_5.pt   <- dataset "best weights"
#   B) fold_1/best.pt ...                   <- gói thẳng từ output run
CKPT_NAMES = ["best_fold_{f}.pt", "best.pt"]

# ---------------------------------------------------------------------------
# KHÔNG hardcode độ sâu. Kaggle mount ở `/kaggle/input/datasets/<user>/<slug>/...`
# chứ không phải `/kaggle/input/<slug>/...` như mọi notebook trước giả định, và độ
# sâu đó có thể đổi tiếp. Dò theo TÊN FILE mốc, sâu bao nhiêu cũng thấy. Đây là lần
# thứ tư sửa cùng một lớp lỗi (S-081 → S-084); nguyên nhân gốc luôn là một giả định
# về hình dạng đường dẫn.
#
# MỘT lượt `os.walk` duy nhất thu hết mọi thứ cần. Không dùng nhiều `rglob` riêng:
# dataset gốc là 83.7GB / ~4000 file trên ổ mạng, và mỗi `rglob` là một lượt duyệt
# toàn cây — 11 lượt thì chờ rất lâu mà chẳng được gì thêm.
# ---------------------------------------------------------------------------
import json as _json
import os as _os
import re as _re

_cfg_data = load_yaml(REPO / "configs" / "data.yaml")
_ann_name = Path(_cfg_data["annotation_rel"]).name

interesting = {}       # thư mục -> số .npz/.pt/meta, để in bảng chẩn đoán
meta_paths = []        # cache_meta.json tìm được
ckpt_paths = []        # mọi file .pt tên best*.pt
_ann = []              # file annotation của dữ liệu gốc

for dirpath, dirnames, filenames in _os.walk(INPUT_ROOT):
    dirnames[:] = [x for x in dirnames if x not in (".cache", ".git")]  # rác tải HF
    here = {"npz": 0, "pt": 0, "meta": 0}
    for name in filenames:
        full = Path(dirpath) / name
        if name.endswith(".npz"):
            here["npz"] += 1
        elif name.endswith(".pt"):
            here["pt"] += 1
            if name.startswith("best"):
                ckpt_paths.append(full)
        elif name == "cache_meta.json":
            here["meta"] += 1
            meta_paths.append(full)
        elif name == _ann_name:
            _ann.append(full)
    if any(here.values()):
        interesting[Path(dirpath)] = here


def read_caches(paths):
    out = []
    for p in sorted(paths):
        try:
            out.append((p.parent, _json.loads(p.read_text("utf-8"))))
        except Exception as exc:  # noqa: BLE001 - chỉ để báo cáo, không nuốt lỗi thật
            out.append((p.parent, {"__loi__": repr(exc)}))
    return out


def matches_e4(meta):
    return all(meta.get(k) == v for k, v in E4_KEYS.items())


def pick_checkpoints(paths, folds):
    """{fold: đường dẫn}. Ưu tiên `best_fold_N.pt`; `best.pt` thì suy fold từ thư
    mục cha (`fold_3/best.pt`). Không suy được thì bỏ, không đoán bừa."""
    out = {}
    for fold in folds:
        hits = [p for p in sorted(paths) if p.name == f"best_fold_{fold}.pt"]
        if not hits:
            hits = [
                p for p in sorted(paths)
                if p.name == "best.pt"
                and (m := _re.search(r"fold_?(\d+)", p.parent.name))
                and int(m.group(1)) == fold
            ]
        if hits:
            out[fold] = hits[0]
    return out


print(f"=== Thư mục có dữ liệu dưới {INPUT_ROOT} ===")
for d in sorted(interesting)[:25]:
    c = interesting[d]
    print(f"  {d}\n      {' · '.join(f'{c[k]} {k}' for k in ('npz', 'pt', 'meta') if c[k])}")
if not interesting:
    print("  (trống — chưa mount dataset nào có .npz/.pt)")

print(f"\n=== Dữ liệu gốc ({_ann_name}) ===")
for p in sorted(_ann)[:5]:
    print(f"  ✓ {p.parent.parent}")
if not _ann:
    print("  KHÔNG thấy — chỉ cần nếu phải build cache (xem ngay dưới)")

caches = read_caches(meta_paths)
print(f"\n=== {len(caches)} cache có cache_meta.json ===")
for path, meta in caches:
    mark = "✓ E4" if matches_e4(meta) else "  --"
    print(
        f"  {mark}  {path}\n"
        f"        crop={meta.get('crop_mode')} size={meta.get('target_size')} "
        f"align={meta.get('align_phases')}"
    )

e4 = [p for p, m in caches if matches_e4(m)]
if e4:
    CACHE_DIR = e4[0]
    BUILD_NEEDED = False
else:
    # Thư mục có nhiều .npz nhưng KHÔNG có meta: không dùng được, và phải nói rõ vì sao.
    # Hình dạng mảng cho biết target_size, nhưng KHÔNG cho biết `align_phases` —
    # E3 (reference) và E4 (per_phase) có cùng shape [8,112,112,32]. Nhận nhầm E3
    # thành E4 sẽ cho ra một bảng kết quả sai mà trông hoàn toàn hợp lý.
    for d, c in sorted(interesting.items()):
        if c["npz"] > 100 and not c["meta"]:
            print(f"\n⚠ {d} có {c['npz']} file .npz nhưng KHÔNG có cache_meta.json.")
            print("  Không dùng được: shape cho biết target_size nhưng KHÔNG phân biệt được")
            print("  E3 (align=reference) với E4 (align=per_phase) — hai cái cùng shape.")
    BUILD_NEEDED = True
    CACHE_DIR = Path("/kaggle/working/cache_e4")
    if _ann:
        print("\n=> sẽ BUILD lại cache E4 (~26 phút). Dữ liệu gốc đã có ✓")
    else:
        print(
            "\n=> CẦN BUILD cache E4 nhưng CHƯA MOUNT dữ liệu gốc.\n"
            f"   Mount dataset chứa {_cfg_data['annotation_rel']} "
            f"(ứng viên: {_cfg_data.get('data_root_candidates')}),\n"
            "   rồi chạy lại từ cell này."
        )

CKPTS = pick_checkpoints(ckpt_paths, FOLDS)
thieu = [f for f in FOLDS if f not in CKPTS]
assert not thieu, (
    f"không thấy checkpoint cho fold {thieu}.\n"
    f"Đã dò theo tên {CKPT_NAMES} ở MỌI độ sâu dưới {INPUT_ROOT}.\n"
    f"Tìm được: { {f: str(p) for f, p in CKPTS.items()} }"
)
print("\ncache:      ", CACHE_DIR, "(CHƯA CÓ — sẽ build ở cell dưới)" if BUILD_NEEDED else "")

# 5 file cùng kiến trúc nên cùng kích thước — kích thước KHÔNG chứng minh chúng khác
# nhau. Băm để chắc không phải một file bị chép 5 lần với 5 cái tên.
import hashlib

print("checkpoint:")
digests = {}
for f in FOLDS:
    p = CKPTS[f]
    h = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
    digests[f] = h
    print(f"  fold {f}: {p.name}  {p.stat().st_size / 2**20:.1f} MB  sha256 {h}")
assert len(set(digests.values())) == len(FOLDS), f"có checkpoint trùng nhau: {digests}"

# Đối chiếu với mã băm đo ở máy local (WORKLOG S-081). Khác => file khác bản.
LOCAL_SHA = {
    1: "2e1f3e1ad477ad59", 2: "30a8eb9ee221d453", 3: "00c133e031bdf8fe",
    4: "3fe18f1eb3de4431", 5: "d61cc7ed94b8ebf0",
}
lech = {f: (digests[f], LOCAL_SHA[f]) for f in FOLDS if f in LOCAL_SHA and digests[f] != LOCAL_SHA[f]}
if lech:
    print(f"\n⚠ mã băm khác bản local: {lech}")
    print("  Không tự dừng — nhưng nếu bạn không cố ý đổi checkpoint thì hãy dừng lại xem.")

## 1b. Build cache E4 nếu chưa có

Chỉ chạy khi cell trên không tìm thấy cache E4 nào. Build lại **cho ra đúng cùng dữ
liệu** — pipeline tiền xử lý tất định, `set_seed` chỉ ảnh hưởng train.

`resolve_data_root` tự lùng dataset LLD-MMRI gốc dưới `/kaggle/input` bằng cách tìm
file annotation, nên không cần khai đường dẫn. Nếu chưa mount dataset gốc thì cell
này sẽ báo rõ chứ không build ra cache rỗng.

In [ ]:
if BUILD_NEEDED:
    from src.utils.io import resolve_data_root

    # Data root khai ở configs/data.yaml, KHÔNG ở preprocess_*.yaml — file preprocess
    # chỉ có tham số tiền xử lý. `build_cache` cũng đọc data.yaml (xem hàm main của
    # nó). Kiểm trước ở đây chỉ để fail nhanh, thay vì chết giữa job 26 phút.
    cfg_data = load_yaml(REPO / "configs" / "data.yaml")
    try:
        data_root = resolve_data_root(cfg_data)
    except Exception as exc:
        # RuntimeError chứ không SystemExit: SystemExit làm IPython lỗi khi dựng
        # traceback và che mất thông báo thật bằng một trang lỗi của chính nó.
        data_root, exc_msg = None, str(exc)
    else:
        exc_msg = None

    # `resolve_data_root` trả về `config['data_root']` mà KHÔNG xác minh khi mọi cách
    # dò đều trượt (src/utils/io.py:219-225). Trên Kaggle nó sẽ là `data/lldmmridataset`
    # tương đối, không tồn tại — và job 26 phút sẽ chết giữa chừng. Xác minh ở đây.
    ann = (data_root / cfg_data["annotation_rel"]) if data_root else None
    if ann is None or not ann.exists():
        raise RuntimeError(
            f"Không tìm thấy dữ liệu LLD-MMRI gốc.\n"
            f"  resolve_data_root -> {data_root}"
            + (f" (lỗi: {exc_msg})" if exc_msg else f", nhưng {ann} không tồn tại")
            + f"\n  Cần mount dataset chứa {cfg_data['annotation_rel']}.\n"
            f"  Ứng viên khai trong configs/data.yaml: {cfg_data.get('data_root_candidates')}"
        ) from None
    print("data root:", data_root, "✓")

    os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
    rc = subprocess.run(
        [sys.executable, "-m", "src.preprocess.build_cache",
         "--config", "configs/preprocess_e4.yaml"],
        cwd=REPO,
    ).returncode
    assert rc == 0, "build cache thất bại"
    print("build xong:", CACHE_DIR)
else:
    print("bỏ qua build — đã có cache E4:", CACHE_DIR)

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

## Cổng A ⚠️ — cache có đúng là E4 không

Chạy MC-dropout trên cache của E1 hay E3 sẽ **không báo lỗi gì cả**, chỉ lặng lẽ cho ra
số sai.

In [ ]:
import json

meta = json.loads((Path(os.environ["LLDMMRI_CACHE_DIR"]) / "cache_meta.json").read_text("utf-8"))

# Dùng lại E4_KEYS của cell trên, không chép ra bản thứ hai — hai bản sẽ trôi khỏi nhau.
for key, want in E4_KEYS.items():
    got = meta.get(key)
    assert got == want, f"cache SAI: {key} = {got!r}, cần {want!r}. Đây không phải cache E4."
assert meta["lesion_tight"]["source"] == "mask", "phải cắt theo mask, không phải bbox"

n_npz = len(list(Path(os.environ["LLDMMRI_CACHE_DIR"]).glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498 — cache chưa build xong"
print(f"cache_meta khớp E4 ✓ · {n_npz} ca · commit {meta.get('git_commit')}")

## Cổng B ⚠️⚠️ — model có dropout thật không

Đây là cổng quan trọng nhất của notebook. Nếu model không có lớp Dropout nào thì `K`
lượt forward cho ra `K` kết quả **giống hệt nhau**, epistemic bằng 0 khắp nơi, và bảng
kết quả vẫn in ra bình thường — một chế độ hỏng hoàn toàn im lặng.

In [ ]:
import torch

from src.eval.mc_dropout import count_dropout_modules, enable_dropout
from src.models import build_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

probe = build_model(CFG["model"])
n_drop = count_dropout_modules(probe)
print(f"số lớp Dropout trong model: {n_drop}")
assert n_drop > 0, (
    "model KHÔNG có lớp Dropout nào — MC-dropout sẽ không làm gì cả. "
    "Kiểm model.dropout_prob trong config."
)

# Và kiểm BatchNorm vẫn ở eval sau khi bật dropout (bẫy chính, xem docstring module).
enable_dropout(probe)
bn = [m for m in probe.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]
assert bn and not any(m.training for m in bn), "BatchNorm phải ở eval sau enable_dropout"
print(f"{len(bn)} lớp BatchNorm, tất cả ở eval ✓")
del probe

## 2. Chạy MC-dropout từng fold

`build_loaders` dựng val loader với `shuffle=False`, đúng thứ cần: các pass phải xếp ca
cùng thứ tự. `mc_dropout_predict` tự kiểm điều đó và nổ nếu lệch.

In [ ]:
import time

import numpy as np

from src.eval.mc_dropout import mc_dropout_predict, save_member_probs
from src.eval.selective import uncertainty_decomposition
from src.eval.metrics import macro_f1
from src.train.run import build_loaders

OUT_ROOT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])

# Đối chiếu epoch trong checkpoint với metrics đã biết, để chắc không nạp nhầm fold.
KNOWN_EPOCH = {1: 231, 2: 297, 3: 104, 4: 135, 5: 144}

for fold in FOLDS:
    t0 = time.time()
    _, val_loader, _ = build_loaders(CFG, fold)
    model = build_model(CFG["model"]).to(DEVICE)

    state = torch.load(CKPTS[fold], map_location=DEVICE)
    model.load_state_dict(state["model"])
    epoch = state.get("epoch")
    print(f"\nfold {fold}: nạp checkpoint epoch {epoch} · {len(val_loader.dataset)} ca")
    if fold in KNOWN_EPOCH and epoch != KNOWN_EPOCH[fold]:
        print(
            f"  ⚠ epoch {epoch} khác {KNOWN_EPOCH[fold]} đã ghi ở WORKLOG S-078 — "
            f"có thể đang nạp checkpoint của fold khác. Kiểm tên file."
        )
    if state.get("fold") not in (None, fold):
        raise RuntimeError(f"checkpoint ghi fold={state['fold']} nhưng đang chạy fold {fold}")

    result = mc_dropout_predict(
        model, val_loader, DEVICE, n_passes=N_PASSES,
        amp=bool(CFG["train"].get("amp", True)), seed=CFG.get("seed", 1337),
    )
    out = save_member_probs(OUT_ROOT / f"fold_{fold}" / "mc_dropout.npz", result)

    members = result["member_probs"]
    mean = members.mean(axis=0)
    unc = uncertainty_decomposition(members)
    print(
        f"  macro-F1 (trung bình {N_PASSES} lượt): {macro_f1(result['labels'], mean.argmax(1)):.4f}"
        f" · epistemic TB {unc['epistemic'].mean():.4f}"
        f" · {time.time() - t0:.0f}s -> {out.name}"
    )
    assert unc["epistemic"].max() > 1e-9, (
        f"fold {fold}: epistemic = 0 khắp nơi — dropout không thực sự chạy"
    )
    del model
    torch.cuda.empty_cache()

## 3. Gói mang về

Chỉ `.npz`, rất nhẹ. Ở máy local:

```
runs/E4_cv_results/fold_N/mc_dropout.npz
python -m src.eval.trust --run-dir runs/E4_cv_results --members
```

In [ ]:
import shutil

PACK = Path("/kaggle/working/mc_dropout_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

for d in sorted(OUT_ROOT.glob("fold*")):
    src = d / "mc_dropout.npz"
    if src.exists():
        (PACK / d.name).mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, PACK / d.name / src.name)

total = sum(f.stat().st_size for f in PACK.rglob("*") if f.is_file())
print(f"đã gói {PACK}: {total / 2**20:.2f} MiB")
for f in sorted(PACK.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(PACK)}  {f.stat().st_size / 2**10:.0f} KiB")

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz bản thân là zip; trình giải nén bung đệ quy
  sẽ biến nó thành thư mục và `src.eval.trust` sẽ không thấy (đã dính hai lần, S-078).
""")